# 1-Million Record Memory Explosion: Autopsy and Streaming Solutions
This notebook analyzes memory consumption when loading large datasets. It compares loading all records into a list with streaming records incrementally using generators.


In [1]:
import os
import tracemalloc
from generate_dataset import generate_large_csv
from streaming_solutions import read_all_inefficient, stream_records

# Generate sample dataset if not present
filename = "large_patients.csv"
generate_large_csv(filename, total_rows=100000)  # Use 100k for fast interactive demo
print(f"File size on disk: {os.path.getsize(filename)/(1024*1024):.2f} MB")


Generating 100,000 patient records in 'large_patients.csv'...
Generation complete.
File size on disk: 1.62 MB


### 1. Inefficient List Allocation
Calling `list(reader)` creates hundreds of thousands of dictionary objects in memory at once.


In [2]:
tracemalloc.start()
bad_data = read_all_inefficient(filename)
_, peak_bad = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak Memory Used: {peak_bad / (1024*1024):.2f} MB")
del bad_data


Peak Memory Used: 31.30 MB


### 2. Generator Streaming Solution
Yielding records one at a time keeps memory usage flat at a few kilobytes, regardless of file size.


In [3]:
tracemalloc.start()
count_high_bp = 0
for row in stream_records(filename):
    if int(row["BloodPressure"]) > 140:
        count_high_bp += 1

_, peak_good = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Peak Memory Used: {peak_good / (1024*1024):.4f} MB")
print(f"Hypertensive patients counted: {count_high_bp:,}")


Peak Memory Used: 0.0514 MB
Hypertensive patients counted: 38,000
